In [76]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [77]:
df = pd.read_csv('E:\imbd_movie_sentiment_analysis\IMDB Dataset.csv')

print(df.head())
print(df.shape)


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [78]:
df.info()

df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [79]:
df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

In [80]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [81]:
vocab_size = 10000
max_length = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [82]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_length),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

e:\imbd_movie_sentiment_analysis\imbd\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [83]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [69]:
print(y_train.shape)
print(y_train[:10])
print(np.unique(y_train))

(40000,)
39087    negative
30893    negative
45278    positive
16398    negative
13653    negative
13748    negative
23965    negative
45552    negative
30219    positive
24079    negative
Name: sentiment, dtype: object
['negative' 'positive']


In [84]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 114s 361ms/step - accuracy: 0.5304 - loss: 0.6886 - val_accuracy: 0.6429 - val_loss: 0.6356
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 260ms/step - accuracy: 0.5727 - loss: 0.6685 - val_accuracy: 0.5075 - val_loss: 0.6856
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 77s 246ms/step - accuracy: 0.5650 - loss: 0.6741 - val_accuracy: 0.5441 - val_loss: 0.6928
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 80s 239ms/step - accuracy: 0.6459 - loss: 0.6409 - val_accuracy: 0.5478 - val_loss: 0.6857
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 76s 243ms/step - accuracy: 0.5684 - loss: 0.6691 - val_accuracy: 0.7215 - val_loss: 0.6211
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 75s 240ms/step - accuracy: 0.7313 - loss: 0.5648 - val_accuracy: 0.7126 - val_loss: 0.6042
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 77s 246ms/step - accuracy: 0.7147 - loss: 0.5885 - val_accuracy: 0.7379 - val_loss: 0.5726
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 83s 266ms/step - accuracy: 0.6392 - loss: 

In [85]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.8633 - loss: 0.3640
Test Accuracy: 0.8632000088691711


In [87]:
model.save("movie_sentiment_model.h5")

In [88]:
def predict_sentiment(review):

    seq = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        seq,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    print("\nReview:")
    print(review)

    print("\nConfidence:", round(float(prediction), 4))

    if prediction >= 0.5:
        print("Sentiment: Positive ")
    else:
        print("Sentiment: Negative ")

In [89]:
predict_sentiment(
    "This movie was absolutely fantastic. Acting and story were amazing."
)

predict_sentiment(
    "Worst movie ever. Total waste of time and money."
)


Review:
This movie was absolutely fantastic. Acting and story were amazing.

Confidence: 0.9742
Sentiment: Positive 

Review:
Worst movie ever. Total waste of time and money.

Confidence: 0.0105
Sentiment: Negative 


In [90]:
predict_sentiment("best movie ever")


Review:
best movie ever

Confidence: 0.7574
Sentiment: Positive 


In [91]:
predict_sentiment("worst movie of all time")


Review:
worst movie of all time

Confidence: 0.028
Sentiment: Negative 


In [40]:
predict_sentiment("horrible movie")


Review:
horrible movie

Confidence: 0.0043
Sentiment: Negative 


In [92]:
import joblib

joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']